<a href="https://colab.research.google.com/github/DongAnYu/ai-tutor-rag-system/blob/main/notebooks/Agents_with_OpenAI_Assistants.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
!pip install -q openai==1.59.6

In [22]:
import os

# Set the "OPENAI_API_KEY" in the Python environment. Will be used by OpenAI client later.
# os.environ["OPENAI_API_KEY"] = "[OPENAI_API_KEY]"


from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Math Tutor


In [23]:
from openai import OpenAI

client = OpenAI()

assistant = client.beta.assistants.create(
    name="Math Tutor",
    instructions="You are a personal math tutor. Write and run code to answer math questions.",
    model="gpt-4o",
    tools=[{"type": "code_interpreter"}],
)

In [24]:
thread = client.beta.threads.create()

In [25]:
message = client.beta.threads.messages.create(
    thread_id=thread.id,
    role="user",
    content="I need to solve the equation `3x + 11 = 14`. Can you help me?",
)

In [26]:
run = client.beta.threads.runs.create_and_poll(
    thread_id=thread.id, assistant_id=assistant.id
)

In [27]:
messages = list(client.beta.threads.messages.list(thread_id=thread.id, run_id=run.id))

In [28]:
print(messages[0].content[0].text.value)

The solution to the equation \(3x + 11 = 14\) is \(x = 1\).


# Customer Support


In [29]:
!wget https://personales.unican.es/corcuerp/linux/resources/LinuxCommandLineCheatSheet_1.pdf

--2026-04-21 20:09:21--  https://personales.unican.es/corcuerp/linux/resources/LinuxCommandLineCheatSheet_1.pdf
Resolving personales.unican.es (personales.unican.es)... 193.144.193.111
Connecting to personales.unican.es (personales.unican.es)|193.144.193.111|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 267816 (262K) [application/pdf]
Saving to: ‘LinuxCommandLineCheatSheet_1.pdf’

LinuxCommandLineChe 100%[===================>] 261.54K   530KB/s    in 0.5s    

2026-04-21 20:09:22 (530 KB/s) - ‘LinuxCommandLineCheatSheet_1.pdf’ saved [267816/267816]



In [30]:
from openai import OpenAI

client = OpenAI()

In [31]:
# Create a vector store caled "Financial Statements"
vector_store = client.beta.vector_stores.create(name="Tech Support")

# Ready the files for upload to OpenAI
file_streams = [open("LinuxCommandLineCheatSheet_1.pdf", "rb")]

# Use the upload and poll SDK helper to upload the files, add them to the vector store,
# and poll the status of the file batch for completion.
file_batch = client.beta.vector_stores.file_batches.upload_and_poll(
    vector_store_id=vector_store.id, files=file_streams
)

# You can print the status and the file counts of the batch to see the result of this operation.
print(file_batch.status)
print(file_batch.file_counts)

completed
FileCounts(cancelled=0, completed=1, failed=0, in_progress=0, total=1)


In [32]:
assistant = client.beta.assistants.create(
    name="Tech Support",
    instructions="You are a tech support chatbot. Use the product manual to respond accurately to customer inquiries.",
    model="gpt-4o",
    tools=[{"type": "file_search"}],
    tool_resources={"file_search": {"vector_store_ids": [vector_store.id]}},
)

In [33]:
# Create a thread and attach the file to the message
thread = client.beta.threads.create(
    messages=[
        {
            "role": "user",
            "content": "What 'ls' command do?",
        }
    ]
)

In [34]:
run = client.beta.threads.runs.create_and_poll(
    thread_id=thread.id, assistant_id=assistant.id
)

In [35]:
messages = list(client.beta.threads.messages.list(thread_id=thread.id, run_id=run.id))

print(messages[0].content[0].text.value)

The `ls` command in Linux is used to list files and directories in the current directory. It can be used with various options to modify its behavior, such as `-al` which lists all files in a long listing format providing detailed information【4:0†LinuxCommandLineCheatSheet_1.pdf】.


In [36]:
messages[0].content[0].text.annotations

[FileCitationAnnotation(end_index=279, file_citation=FileCitation(file_id='file-N1YqTwumhZffrn8GU9sM2z'), start_index=241, text='【4:0†LinuxCommandLineCheatSheet_1.pdf】', type='file_citation')]